# Masterclass 1: Multi-Sector Exploratory Spatial Data Analysis (ESDA) & Disease Surveillance
### *Evidence-Based Spatial Intelligence Across Public Health, Disease Epidemiology (Malaria), Geomarketing, Cultural Geography, and Infrastructure*

---

## 1. Introduction & Key Concepts

In classical non-spatial data science, observations are assumed to be **independent and identically distributed (i.i.d.)**. When analyzing geographic units---such as Nigeria's **9,308 administrative wards**---this assumption fundamentally collapses due to **Tobler's First Law of Geography**:

> *"Everything is related to everything else, but near things are more related than distant things."*  
> --- Waldo Tobler (1970)

When governments or enterprises make capital allocation decisions using state or national averages, they fall victim to three major fallacies:
1. **The Fallacy of the State Average:** High aggregate wealth in an urban LGA masks severe, isolated rural deprivation within the same state.
2. **The Spatial Spillover Blindspot:** Interventions in one ward (e.g., establishing a regional wholesale market or hospital) generate positive economic feedback loops into contiguous wards.
3. **The Misallocation Trap:** Deploying resources where competition is saturated while ignoring high-need, high-potential underserved catchments.

---

## 2. Core Sectors Investigated
- **Public Health:** Detecting "Healthcare Deserts" (high population, 0 clinics).
- **Disease Surveillance:** Modeling Malaria parasite prevalence ($Pf\text{PR}_{2-10}$) hotspots.
- **Geomarketing & Retail:** Identifying high-wealth, low-competition commercial retail catchments.
- **Cultural Geography:** Mapping religious institutional distribution and Shannon diversity transition zones.
- **Spatial Topology & Clustering:** Global Moran's $I$ and Anselin Local Moran (LISA) cluster detection.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import libpysal
import esda
from splot.esda import moran_scatterplot, lisa_cluster

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_style('whitegrid')


In [ ]:
DATA_PATH = '../data/processed/nigeria_wards_master.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/nigeria_wards_master.parquet'

print(f"Loading master dataset from: {DATA_PATH}")
gdf = gpd.read_parquet(DATA_PATH)

if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)
else:
    gdf = gdf.to_crs(epsg=4326)

gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.reset_index(drop=True, inplace=True)
print(f"Dataset successfully loaded: {len(gdf):,} administrative wards across {gdf['statename'].nunique()} states.")


In [ ]:
key_cols = [
    'rwi_mean', 'pop_2025_sum', 'population_density_per_sqkm',
    'health_facilities_count', 'markets_count', 'water_points_count',
    'churches_count', 'mosques_count', 'schools_count', 'police_stations_count',
    'malaria_prevalence_pct'
]
summary_table = gdf[key_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
summary_table.columns = ['Count', 'Mean', 'Std Dev', 'Min', 'Median (IQR)', 'Max']
summary_table.round(3)


## 3. Spatial Weights Matrix ($W$): Formalizing Geographic Topology

To compute spatial statistics, we convert continuous polygons into an $n \times n$ spatial connectivity matrix $W$.
Two primary specifications:
1. **Queen Contiguity:** Units $i$ and $j$ are neighbors if they share an edge or a vertex.
2. **$K$-Nearest Neighbors (KNN):** Units $i$ and $j$ are connected based on centroid distance ($k=5$). KNN guarantees no disconnected islands.

Row-Standardization:
$$w_{ij}^* = \frac{w_{ij}}{\sum_{k=1}^n w_{ik}} \quad \implies \quad \sum_{j=1}^n w_{ij}^* = 1$$


In [ ]:
w_queen = libpysal.weights.Queen.from_dataframe(gdf, use_index=False)
w_queen.transform = 'R'

w_knn = libpysal.weights.KNN.from_dataframe(gdf, k=5)
w_knn.transform = 'R'

print(f"Queen Contiguity: {w_queen.n} wards, Islands: {len(w_queen.islands)}, Mean Neighbors: {w_queen.mean_neighbors:.2f}")
print(f"KNN-5 Topology:   {w_knn.n} wards, Islands: {len(w_knn.islands)}, Mean Neighbors: {w_knn.mean_neighbors:.2f}")


## 4. Global Spatial Autocorrelation: Moran's $I$

Moran's $I$ determines whether a variable exhibits spatial clustering, spatial dispersion, or complete spatial randomness ($H_0$).

$$
I = \frac{n}{S_0} \frac{\sum_{i=1}^n \sum_{j=1}^n w_{ij}(y_i - \bar{y})(y_j - \bar{y})}{\sum_{i=1}^n (y_i - \bar{y})^2}
$$
where $S_0 = \sum_{i=1}^n \sum_{j=1}^n w_{ij}$. Under $H_0$, $E[I] = -\frac{1}{n-1} \approx 0$.


In [ ]:
rwi_clean = gdf['rwi_mean'].fillna(gdf['rwi_mean'].median()).values
mi_rwi = esda.Moran(rwi_clean, w_knn, permutations=999)

fig, ax = plt.subplots(figsize=(8, 6))
moran_scatterplot(mi_rwi, ax=ax)
ax.set_title(f"Global Moran's I: Relative Wealth Index (RWI)\nI = {mi_rwi.I:.3f}, z = {mi_rwi.z_sim:.1f} (p < 0.001)", fontsize=13, fontweight='bold')
ax.axhline(0, color='grey', linestyle='--', lw=0.8)
ax.axvline(0, color='grey', linestyle='--', lw=0.8)
plt.tight_layout()
plt.show()

print(f"Global Moran's I (Wealth): {mi_rwi.I:.4f} | z-score: {mi_rwi.z_sim:.2f} | p-value: {mi_rwi.p_sim:.4f}")


## 5. Public Health Spotlight: Detecting "Healthcare Deserts"

Healthcare Deserts are defined as administrative wards where the population exceeds the national median (> 17,000 residents) but has **exactly 0 registered health facilities**.


In [ ]:
pop_med = gdf['pop_2025_sum'].median()
gdf['is_health_desert'] = (gdf['health_facilities_count'] == 0) & (gdf['pop_2025_sum'] > pop_med)
n_deserts = gdf['is_health_desert'].sum()
pop_affected = gdf[gdf['is_health_desert']]['pop_2025_sum'].sum()

print(f"Healthcare Deserts identified: {n_deserts:,} wards ({n_deserts/len(gdf)*100:.1f}% of total).")
print(f"Total vulnerable population living in deserts: {pop_affected:,.0f} residents.")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))

gdf.plot(column='rwi_mean', cmap='viridis', legend=True, ax=ax1,
         legend_kwds={'label': 'Relative Wealth Index (RWI)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. National Relative Wealth Index (RWI)", fontsize=12, fontweight='bold')
ax1.axis('off')

gdf.plot(color='#ececec', edgecolor='#ffffff', linewidth=0.1, ax=ax2)
gdf[gdf['is_health_desert']].plot(color='#d90429', ax=ax2)
ax2.set_title(f"B. Critical Healthcare Deserts (n={n_deserts:,})", fontsize=12, fontweight='bold')
ax2.axis('off')

desert_p = mpatches.Patch(color='#d90429', label=f"Healthcare Deserts (n={n_deserts:,})")
base_p = mpatches.Patch(color='#ececec', label=f"Wards with Health Facilities (n={(~gdf['is_health_desert']).sum():,})")
ax2.legend(handles=[desert_p, base_p], loc='lower left', frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()


## 6. Epidemiological Disease Surveillance: Malaria Transmission

Mapping:
1. **Malaria Parasite Prevalence ($Pf\text{PR}_{2-10}$ %):** High in humid southern basins, seasonal in northern sahel.
2. **Anselin Local Moran Disease Clusters:** Distinguishing endemic transmission hotspots from protected zones.


In [ ]:
if 'malaria_prevalence_pct' in gdf.columns:
    mal_clean = gdf['malaria_prevalence_pct'].fillna(gdf['malaria_prevalence_pct'].median()).values
    mi_mal = esda.Moran(mal_clean, w_knn, permutations=999)
    print(f"Global Moran's I for Malaria: {mi_mal.I:.4f} | z-score: {mi_mal.z_sim:.2f} | p-value: {mi_mal.p_sim:.4f}")
    
    fig, ax = plt.subplots(figsize=(10, 7))
    gdf.plot(column='malaria_prevalence_pct', cmap='YlOrRd', legend=True, ax=ax,
             legend_kwds={'label': 'Malaria Parasite Prevalence (PfPR 2-10 %)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
    ax.set_title("National Malaria Parasite Prevalence across 9,308 Wards", fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


## 7. Commercial Retail Catchment & Expansion Segmentation

Segmenting wards into four strategic commercial quadrants based on Relative Wealth (RWI) and physical Market counts:
- **Tier 1 (Saturated Affluent):** High Wealth, High Markets (Deepen premium assortment).
- **Tier 2 (Prime Expansion Target):** High Wealth, Low Markets (**Prime entry opportunity**).
- **Tier 3 (Informal Commerce Hubs):** Low Wealth, High Markets (High volume, low margin).
- **Tier 4 (Subsistence / Underdeveloped):** Low Wealth, Low Markets (Community retail models).


In [ ]:
rwi_med = gdf['rwi_mean'].median()
mkt_med = gdf['markets_count'].median()

conditions = [
    (gdf['rwi_mean'] >= rwi_med) & (gdf['markets_count'] >= mkt_med),
    (gdf['rwi_mean'] >= rwi_med) & (gdf['markets_count'] < mkt_med),
    (gdf['rwi_mean'] < rwi_med) & (gdf['markets_count'] >= mkt_med),
    (gdf['rwi_mean'] < rwi_med) & (gdf['markets_count'] < mkt_med)
]
choices = [
    "Tier 1: Saturated Affluent (High RWI, High Markets)",
    "Tier 2: Prime Expansion Target (High RWI, Low Markets)",
    "Tier 3: Informal Commerce Hubs (Low RWI, High Markets)",
    "Tier 4: Subsistence / Underdeveloped (Low RWI, Low Markets)"
]
gdf['retail_segment'] = np.select(conditions, choices, default='Unclassified')

fig, ax = plt.subplots(figsize=(11, 7.5))
colors = {
    "Tier 1: Saturated Affluent (High RWI, High Markets)": "#1b4332",
    "Tier 2: Prime Expansion Target (High RWI, Low Markets)": "#52b788",
    "Tier 3: Informal Commerce Hubs (Low RWI, High Markets)": "#e76f51",
    "Tier 4: Subsistence / Underdeveloped (Low RWI, Low Markets)": "#d8d8d8"
}
for seg, col in colors.items():
    sub = gdf[gdf['retail_segment'] == seg]
    sub.plot(color=col, ax=ax, label=seg, linewidth=0.1, edgecolor='white')

ax.set_title("Commercial Retail Catchment Strategy Map (9,308 Wards)", fontsize=13, fontweight='bold')
ax.axis('off')
patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()


## 8. Cultural Geography: Faith Infrastructure & Religious Coexistence

Nigeria exhibits a distinct North-South religious institutional divergence, with a critical pluralistic Middle Belt.
We calculate:
1. **Church vs. Mosque Share:** $S_i = \frac{\text{Churches}_i}{\text{Churches}_i + \text{Mosques}_i + 0.01}$
2. **Shannon Entropy (Cultural Diversity Index):**
$$H_i = -\sum_{k \in \{c, m\}} p_{ik} \ln(p_{ik})$$


In [ ]:
gdf['church_share'] = gdf['churches_count'] / (gdf['churches_count'] + gdf['mosques_count'] + 0.01)
p_c = gdf['churches_count'] / (gdf['churches_count'] + gdf['mosques_count'] + 1e-6)
p_m = gdf['mosques_count'] / (gdf['churches_count'] + gdf['mosques_count'] + 1e-6)
h_c = np.where(p_c > 0, p_c * np.log(p_c + 1e-12), 0)
h_m = np.where(p_m > 0, p_m * np.log(p_m + 1e-12), 0)
gdf['religious_diversity_idx'] = -(h_c + h_m)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7.5))
gdf.plot(column='church_share', cmap='coolwarm', legend=True, ax=ax1,
         legend_kwds={'label': 'Church Share (0=Mosque Dominant, 1=Church Dominant)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. Religious Infrastructure Spatial Sorting", fontsize=12, fontweight='bold')
ax1.axis('off')

gdf.plot(column='religious_diversity_idx', cmap='magma', legend=True, ax=ax2,
         legend_kwds={'label': 'Shannon Entropy Diversity Index', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax2.set_title("B. Cultural Transition Zones (High Coexistence)", fontsize=12, fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.show()


## 9. Local Spatial Autocorrelation: Anselin LISA Cluster Detection

While Global Moran's $I$ proves national clustering, Anselin Local Moran ($I_i$) locates each cluster:
$$
I_i = \frac{z_i}{s^2} \sum_{j=1}^n w_{ij} z_j
$$
Categories:
- **High-High (Hotspots):** High wealth surrounded by high wealth.
- **Low-Low (Coldspots):** Severe structural asset poverty clusters.
- **High-Low (Outliers):** Affluent islands surrounded by poverty.
- **Low-High (Outliers):** Pockets of poverty inside metropolitan cores.


In [ ]:
lm_rwi = esda.Moran_Local(rwi_clean, w_knn, transformation='r', permutations=999, seed=42)
sig = lm_rwi.p_sim < 0.05

hotspots = (lm_rwi.q == 1) & sig
coldspots = (lm_rwi.q == 3) & sig
high_low = (lm_rwi.q == 4) & sig
low_high = (lm_rwi.q == 2) & sig

gdf['lisa_cluster'] = 'Not Significant'
gdf.loc[hotspots, 'lisa_cluster'] = 'High-High (Hotspot)'
gdf.loc[coldspots, 'lisa_cluster'] = 'Low-Low (Coldspot)'
gdf.loc[high_low, 'lisa_cluster'] = 'High-Low (Outlier)'
gdf.loc[low_high, 'lisa_cluster'] = 'Low-High (Outlier)'

lisa_colors = {
    'Not Significant': '#f0f0f0',
    'High-High (Hotspot)': '#d90429',
    'Low-Low (Coldspot)': '#0077b6',
    'High-Low (Outlier)': '#f77f00',
    'Low-High (Outlier)': '#90e0ef'
}

fig, ax = plt.subplots(figsize=(12, 8))
for ctype, color in lisa_colors.items():
    sub = gdf[gdf['lisa_cluster'] == ctype]
    if len(sub) > 0:
        sub.plot(color=color, ax=ax, label=f"{ctype} (n={len(sub):,})", linewidth=0.1, edgecolor='white')

ax.set_title("Anselin Local Moran's I (LISA) Relative Wealth Clusters", fontsize=13, fontweight='bold')
ax.axis('off')
lisa_patches = [mpatches.Patch(color=c, label=f"{l} (n={(gdf['lisa_cluster'] == l).sum():,})") for l, c in lisa_colors.items()]
ax.legend(handles=lisa_patches, loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


## 10. Strategic Multi-Sector Decision Playbook

| Spatial Quadrant | Public Health Action | Commercial Geomarketing Action | Disease & Vector Control Action |
| :--- | :--- | :--- | :--- |
| **High-High (Hotspot)** | Private hospital licensing; premium health maintenance organizations (HMOs). | Flagship retail stores, modern supermarkets, agency banking networks. | Routine urban surveillance; indoor pest management. |
| **Low-Low (Coldspot)** | Subsidized primary health clinics, mobile clinical vans, free maternal kits. | Low-ticket consumer products, sachet goods, micro-finance depots. | Intensive Indoor Residual Spraying (IRS) and universal bed net distribution. |
| **High-Low (Outlier)** | Regional referral hospital; serves as medical hub for surrounding rural areas. | Regional wholesale depot, cash-and-carry warehouse. | Regional diagnostic laboratory and antimalarial medication stockpile. |
| **Low-High (Outlier)** | Municipal utility expansion; connecting excluded slum communities to city care. | Community retail shops, commuter transit sales points. | Environmental drainage remediation and larvicide application. |


## Primary Data Sources & Key References

### Primary Geospatial Data Sources
- **Administrative Ward Boundaries:** GRID3 Nigeria Admin-3 Wards (9,308 polygons): [https://grid3.gov.ng/datasets/nigeria/administrative-boundaries](https://grid3.gov.ng/datasets/nigeria/administrative-boundaries)
- **Relative Wealth Index (RWI):** Meta AI Research & UC Berkeley micro-wealth estimates: [https://data.humdata.org/dataset/relative-wealth-index](https://data.humdata.org/dataset/relative-wealth-index)
- **Demographic Population Counts:** WorldPop 2025 Gridded Population Projections: [https://hub.worldpop.org/geodata/listing?id=29](https://hub.worldpop.org/geodata/listing?id=29)
- **Points of Interest Registries:** GRID3 Nigeria Health Clinics, Markets, Water Points, Police, Religious Centers: [https://grid3.gov.ng/datasets](https://grid3.gov.ng/datasets)
- **Disease Epidemiology:** Malaria Atlas Project (MAP) Plasmodium falciparum $Pf\text{PR}_{2-10}$: [https://malariaatlas.org/](https://malariaatlas.org/)
- **Electoral Infrastructure:** INEC Polling Units Location Registry: [https://irev.inecnigeria.org](https://irev.inecnigeria.org)

### Methodological References & Literature
1. **Anselin, L. (1988).** *Spatial Econometrics: Methods and Models*. Kluwer Academic Publishers.
2. **Anselin, L. (1995).** Local Indicators of Spatial Association -- LISA. *Geographical Analysis*, 27(2), 93-115.
3. **Chi, G., Fang, H., Chatterjee, S., & Blumenstock, J. E. (2022).** Micro-estimate of wealth for all low- and middle-income countries. *PNAS*, 119(3), e2113658119.
4. **Rey, S. J., & Anselin, L. (2007).** PySAL: A Python library for spatial analytical methods. *The Review of Regional Studies*, 37(1), 5-27.
5. **Tobler, W. R. (1970).** A computer movie simulating urban growth in the Detroit region. *Economic Geography*, 46(sup1), 234-240.
6. **Weiss, D. J., et al. (2019).** Mapping the global prevalence, incidence, and mortality of Plasmodium falciparum, 2000-17. *The Lancet*, 394(10195), 322-331.
